In [5]:
import numpy as np
import pandas as pd
from scipy.optimize import brentq

# Retirement Subsidy for Diesel ICEV

[Korea plans to subsidize the early retirement of diesel cars of grade 4 and grade 5 and that of gasoline cars of grade 5.](https://www.mecar.or.kr/lpm/info/oldCarEarlyScrapping.do) In 2019.6, the number of enrolled diesel cars was 9,989,629. That of gasoline was 11,140,964. [That of grade 3 diesel cars was 6,200,340. That of grade 4 diesel cars was 1,350,352. And that of grade 5 was 2,438,937.](https://www.korea.kr/briefing/pressReleaseView.do?newsId=156338348#pressRelease) That of grade 5 gasoline cars was 31,116. [By the Korean government, the retirement rate for grade 4 diesel cars increased by 9.6%p.](https://www.hankyung.com/article/202402181041i) This effect is assumed to be applied for grade 4 and grade 5 diesel and grade 5 gasoline. Refer to 2022 stats [here](https://ctis.re.kr/en/selectBbsNttView.do?key=1574&bbsNo=312&nttNo=1129021&searchCtgry=&searchCnd=all&searchKrwd=&pageIndex=5&searchBbsType=&chgPage=).

In [28]:
yearly_effect = (0.141 - 0.045) * (1350352 + 2438937) / (11140964 + 9989629)
yearly_effect

0.01721540630686512

In [29]:
periodic_effect = yearly_effect * 5
periodic_effect

0.0860770315343256

GCAM employs [S-curve shutdown function]((https://jgcri.github.io/gcam-doc/en_technologies.html)) as below for deciding stock turnover rates.

$$\text{output}=\frac{1}{1+\exp{s*(t-h)}}$$

where $s$ and $h$ are predetermined parameters, and $t$ denotes time. The table below presents the basic parameter assumptions by vehicle type related to stock turnover in GCAM. These parameters are used to calculate the default turnover rates, which are then adjusted to reflect the effect of the retirement subsidy. Detailed implementation steps are provided below.

```xml
<?xml version="1.0" encoding="UTF-8"?><scenario>
    <world>
        <global-technology-database>
            <location-info sector-name="trn_pass_road_LDV_4W" subsector-name="Car">
                <tranTechnology name="Liquids">
                    <period year="2015">
                        <lifetime>25</lifetime>
                        <s-curve-shutdown-decider name="s-curve">
                            <steepness>0.218</steepness>
                            <half-life>11</half-life>
                        </s-curve-shutdown-decider>
                    </period>
                </tranTechnology>
                
            </location-info>

            <location-info sector-name="trn_pass_road_LDV_4W" subsector-name="Large Car and Truck">
                
                <tranTechnology name="Liquids">
                    <period year="2015">
                        <lifetime>25</lifetime>
                        <s-curve-shutdown-decider name="s-curve">
                            <steepness>0.23</steepness>
                            <half-life>11</half-life>
                        </s-curve-shutdown-decider>
                    </period>
                </tranTechnology>
                
            </location-info>

            <location-info sector-name="trn_freight_road" subsector-name="Medium truck">
                
                <tranTechnology name="Liquids">
                    <period year="2015">
                        <lifetime>20</lifetime>
                        <s-curve-shutdown-decider name="s-curve">
                            <steepness>0.193</steepness>
                            <half-life>10</half-life>
                        </s-curve-shutdown-decider>
                    </period>
                </tranTechnology>
                
            </location-info>
        </global-technology-database>
    </world>
</scenario>

```

In [30]:
def s_curve(t, s, h):
    return 1 / (1+np.exp(s*(t-h)))

In [31]:
def make_objective(base, t, h, target_diff):
    def f(x):
        return base - s_curve(t, x, h) - target_diff
    return f


In [32]:
base = s_curve(20, 0.218, 11)

obj = make_objective(base, t=20, h=11, target_diff=periodic_effect)

from scipy.optimize import brentq
x = brentq(obj, 0.22, 0.5)
print(x)

0.36158561941302586


In [33]:
base = s_curve(20, 0.23, 11)

obj = make_objective(base, t=20, h=11, target_diff=periodic_effect)

from scipy.optimize import brentq
x = brentq(obj, 0.22, 0.5)
print(x)

0.40272220007676046


In [34]:
base = s_curve(20, 0.193, 10)

obj = make_objective(base, t=20, h=10, target_diff=periodic_effect)

from scipy.optimize import brentq
x = brentq(obj, 0.22, 0.5)
print(x)

0.31606534557869065


In [61]:
yearly_effect = (0.141 - 0.045) * (1350352 + 2438937 + 6200340) / (11140964 + 9989629)
yearly_effect

0.04538464131129684

In [62]:
periodic_effect = yearly_effect * 5
periodic_effect

0.22692320655648418

In [65]:
s_curve(15, 0.218, 11)

0.2948383140857415

In [70]:
base = s_curve(15, 0.218, 11)

obj = make_objective(base, t=15, h=11, target_diff=periodic_effect)
x = brentq(obj, 0.22, 0.9)
print(x)

0.6547913475787138


In [71]:
base = s_curve(15, 0.23, 11)

obj = make_objective(base, t=15, h=11, target_diff=periodic_effect)
x = brentq(obj, 0.22, 0.9)
print(x)

0.6967318885222727


In [72]:
base = s_curve(15, 0.193, 10)

obj = make_objective(base, t=15, h=10, target_diff=periodic_effect)
x = brentq(obj, 0.22, 0.9)
print(x)

0.5933318768517508
